In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
pip install datasets openai

creating the tasks bank

In [ ]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from openai import OpenAI
from google.colab import drive

# 1. Mount Google Drive to access all files
drive.mount('/content/drive', force_remount=True)

# 2. Initialize OpenAI Client (Ensure your API Key is correct)
client = OpenAI(api_key='YOUR_OPENAI_API_KEY')

# 3. Define paths
words_path = '/content/drive/My Drive/LLM/ami_public_manual_1/words'
output_path = '/content/drive/My Drive/LLM/Global_Task_Bank.txt'

def create_global_task_bank():
    # A. Get IDs of ALL unique meetings available in the folder
    all_files = sorted([f for f in os.listdir(words_path) if f.endswith('.words.xml')])
    meeting_ids = sorted(list(set([f.split('.')[0] for f in all_files])))

    print(f" Found {len(meeting_ids)} meetings in total. Starting extraction...")

    full_task_list = []
    current_chunk_text = ""
    chunk_count = 1

    # B. Iterate through every single meeting
    for i, mid in enumerate(meeting_ids):
        meeting_content = ""
        for speaker in ['A', 'B', 'C', 'D']:
            f_path = os.path.join(words_path, f"{mid}.{speaker}.words.xml")
            if os.path.exists(f_path):
                try:
                    tree = ET.parse(f_path)
                    words = [w.text for w in tree.getroot().findall('w') if w.text]
                    meeting_content += " ".join(words) + " "
                except Exception:
                    continue

        current_chunk_text += meeting_content + "\n\n"

        # C. If the accumulated text is large enough (~15,000 characters), process it
        if len(current_chunk_text) > 15000 or i == len(meeting_ids) - 1:
            print(f" Processing Chunk {chunk_count} via OpenAI...")
            try:
                response = client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=[
                        {
                            "role": "system",
                            "content": "You are a data analyst. Extract a bulleted list of action items and tasks from this transcript segment. Only return the tasks in English."
                        },
                        {"role": "user", "content": current_chunk_text}
                    ]
                )
                full_task_list.append(response.choices[0].message.content)
            except Exception as e:
                print(f" Error in chunk {chunk_count}: {e}")

            # Reset for next chunk
            current_chunk_text = ""
            chunk_count += 1

    # D. Consolidate all tasks and save to a single file
    final_task_bank = "\n".join(full_task_list)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(final_task_bank)

    print(f" Success! Global Task Bank saved to: {output_path}")
    return final_task_bank

# Run the global extraction
global_bank = create_global_task_bank()

Mounted at /content/drive
 Found 171 meetings in total. Starting extraction...
 Processing Chunk 1 via OpenAI...
 Processing Chunk 2 via OpenAI...
 Processing Chunk 3 via OpenAI...
 Processing Chunk 4 via OpenAI...
 Processing Chunk 5 via OpenAI...
 Processing Chunk 6 via OpenAI...
 Processing Chunk 7 via OpenAI...
 Processing Chunk 8 via OpenAI...
 Processing Chunk 9 via OpenAI...
 Processing Chunk 10 via OpenAI...
 Processing Chunk 11 via OpenAI...
 Error in chunk 11: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 19343 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
 Processing Chunk 12 via OpenAI...
 Processing Chunk 13 via OpenAI...
 Processing Chunk 14 via OpenAI...
 Processing Chunk 15 via OpenAI...
 Processing Chunk 16 via OpenAI...
 Error in chunk 16: Error code: 400 - {'error': {'message': "This model's

translating the bank to hebrish

In [ ]:
import pandas as pd
from openai import OpenAI
from google.colab import drive
import os

# 1. Mount Drive and Initialize Client
drive.mount('/content/drive', force_remount=True)
client = OpenAI(api_key='YOUR_OPENAI_API_KEY')

# 2. Define Paths
input_path = '/content/drive/My Drive/LLM/Global_Task_Bank.txt'
output_path = '/content/drive/My Drive/LLM/Heblish_Task_Bank.csv'

def translate_bank_to_heblish():
    # A. Read the English Task Bank
    if not os.path.exists(input_path):
        print(" Task Bank file not found!")
        return

    with open(input_path, 'r', encoding='utf-8') as f:
        tasks = [line.strip() for line in f.readlines() if line.strip() and not line.startswith('#')]

    print(f" Found {len(tasks)} tasks. Starting translation to Heblish...")

    heblish_tasks = []

    # B. Process in batches to save time and API calls
    # We ask OpenAI to translate the list while maintaining the context of a tech office
    for i in range(0, len(tasks), 10):
        batch = tasks[i:i+10]
        batch_text = "\n".join(batch)

        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {
                        "role": "system",
                        "content": """You are an Israeli tech lead.
                        Translate these English tasks into natural 'Heblish' (Hebrew with tech English terms).
                        - Use Hebrew script (אותיות עבריות).
                        - Keep professional terms in English (e.g., UI, Budget, Feature, Ticket, Sync, Design).
                        - Make it sound like a real person talking in an office.
                        - Return exactly one line per task."""
                    },
                    {"role": "user", "content": batch_text}
                ],
                temperature=0.5
            )

            translated_batch = response.choices[0].message.content.strip().split('\n')
            heblish_tasks.extend(translated_batch)
            print(f" Translated {len(heblish_tasks)} / {len(tasks)} tasks...")

        except Exception as e:
            print(f" Error in batch: {e}")

    # C. Save as a CSV for easy use in training
    df = pd.DataFrame({
        'original_english': tasks[:len(heblish_tasks)],
        'heblish_task': heblish_tasks
    })

    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Finished! Heblish Task Bank saved to: {output_path}")

# Run the translation
translate_bank_to_heblish()

Mounted at /content/drive
🔄 Found 2033 tasks. Starting translation to Heblish...
 Translated 10 / 2033 tasks...
 Translated 20 / 2033 tasks...
 Translated 30 / 2033 tasks...
 Translated 40 / 2033 tasks...
 Translated 50 / 2033 tasks...
 Translated 60 / 2033 tasks...
 Translated 70 / 2033 tasks...
 Translated 80 / 2033 tasks...
 Translated 90 / 2033 tasks...
 Translated 100 / 2033 tasks...
 Translated 110 / 2033 tasks...
 Translated 120 / 2033 tasks...
 Translated 130 / 2033 tasks...
 Translated 140 / 2033 tasks...
 Translated 150 / 2033 tasks...
 Translated 160 / 2033 tasks...
 Translated 170 / 2033 tasks...
 Translated 180 / 2033 tasks...
 Translated 189 / 2033 tasks...
 Translated 198 / 2033 tasks...
 Translated 208 / 2033 tasks...
 Translated 218 / 2033 tasks...
 Translated 228 / 2033 tasks...
 Translated 238 / 2033 tasks...
 Translated 248 / 2033 tasks...
 Translated 258 / 2033 tasks...
 Translated 268 / 2033 tasks...
 Translated 278 / 2033 tasks...
 Translated 288 / 2033 tasks...


creating the meetings

In [ ]:
import pandas as pd
from openai import OpenAI
import os
import random
import json
import re

# Setup Client and Paths
client = OpenAI(api_key='YOUR_OPENAI_API_KEY')

input_path = '/content/drive/My Drive/LLM/Heblish_Task_Bank.csv'
output_folder = '/content/drive/My Drive/LLM/Generated_Meetings'

def generate_realistic_meetings():
    if not os.path.exists(input_path):
        print(f"Error: Heblish Task Bank not found at {input_path}")
        return

    df_bank = pd.read_csv(input_path)
    tasks_pool = df_bank['heblish_task'].tolist()

    num_meetings = 250
    print(f"Starting generation of {num_meetings} meetings with JSON action items...")

    for i in range(num_meetings):
        num_tasks = random.randint(1, 4)
        selected_tasks = random.sample(tasks_pool, num_tasks)

        meeting_length = random.randint(15, 35)

        prompt = f"""
        Create a realistic Israeli tech office dialogue in Hebrew ({meeting_length} lines).
        The dialogue must include professional tech terms in English.

        TASKS TO DISCUSS AND COMMIT TO:
        {chr(10).join([f"- {t}" for t in selected_tasks])}

        STRICT RULES:
        1. Format dialogue: Speaker: Text | Label (1 for commitment lines, 0 otherwise).
        2. AFTER the dialogue, add a section starting with 'JSON_START' and ending with 'JSON_END'.
        3. Inside, provide a VALID JSON object with this exact structure:
           {{
             "action_items": [
               {{ "assignee": "Name", "action": "Task description", "deadline": "Date/Time" }}
             ]
           }}
        4. Use Hebrew for names and deadlines.
        """

        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=2500
            )

            full_content = response.choices[0].message.content.strip()


            dialogue_part = full_content.split('JSON_START')[0].strip()
            current_meeting_rows = []
            for line in dialogue_part.split('\n'):
                if '|' in line and ':' in line:
                    try:
                        speaker_text, label = line.split('|')
                        speaker, text = speaker_text.split(':', 1)
                        current_meeting_rows.append({
                            'utterance': text.strip(),
                            'label': int(label.strip()),
                            'speaker': speaker.strip(),
                            'meeting_id': f"Meeting_{i}"
                        })
                    except:
                        continue


            json_match = re.search(r'JSON_START(.*?)JSON_END', full_content, re.DOTALL)

            if json_match:
                try:
                    json_str = json_match.group(1).strip()
                    json_data = json.loads(json_str)

                    json_filename = f"Meeting_{i}_tasks.json"
                    with open(os.path.join(output_folder, json_filename), 'w', encoding='utf-8') as f:
                        json.dump(json_data, f, ensure_ascii=False, indent=4)
                    json_status = "JSON saved"
                except json.JSONDecodeError:
                    json_status = "JSON error (Invalid format)"
            else:
                json_status = "JSON not found (Truncated?)"


            if current_meeting_rows:
                df_meeting = pd.DataFrame(current_meeting_rows)
                csv_filename = f"Meeting_{i}.csv"
                df_meeting.to_csv(os.path.join(output_folder, csv_filename), index=False, encoding='utf-8-sig')
                print(f"Done: Meeting_{i} -> {json_status}")
            else:
                print(f"Skipped Meeting_{i}: No dialogue generated.")

        except Exception as e:
            print(f"Error in meeting {i}: {e}")

    print(f"\nFinished! Files are in: {output_folder}")


generate_realistic_meetings()

Starting generation of 250 meetings with JSON action items...
Done: Meeting_0 -> JSON saved
Done: Meeting_1 -> JSON saved
Done: Meeting_2 -> JSON saved
Done: Meeting_3 -> JSON saved
Done: Meeting_4 -> JSON saved
Done: Meeting_5 -> JSON saved
Done: Meeting_6 -> JSON saved
Done: Meeting_7 -> JSON saved
Done: Meeting_8 -> JSON saved
Done: Meeting_9 -> JSON error (Invalid format)
Done: Meeting_10 -> JSON error (Invalid format)
Done: Meeting_11 -> JSON saved
Done: Meeting_12 -> JSON saved
Done: Meeting_13 -> JSON error (Invalid format)
Done: Meeting_14 -> JSON saved
Done: Meeting_15 -> JSON saved
Done: Meeting_16 -> JSON saved
Done: Meeting_17 -> JSON saved
Done: Meeting_18 -> JSON saved
Done: Meeting_19 -> JSON saved
Done: Meeting_20 -> JSON saved
Done: Meeting_21 -> JSON saved
Done: Meeting_22 -> JSON saved
Done: Meeting_23 -> JSON saved
Done: Meeting_24 -> JSON saved
Done: Meeting_25 -> JSON saved
Done: Meeting_26 -> JSON saved
Done: Meeting_27 -> JSON saved
Done: Meeting_28 -> JSON save